In [1]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path
import time
import csv
import re


from __future__ import annotations
import asyncio
import nest_asyncio
import random
from dataclasses import dataclass
from typing import Any

from openai import AsyncOpenAI, APIConnectionError, APIStatusError, RateLimitError
from tqdm.asyncio import tqdm

In [2]:
import logging
import sys
from pathlib import Path
from datetime import datetime
from io import StringIO

# ─────────────────────────────────────────────
# Монтирование Google Drive
# ─────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    LOG_DIR = Path("/content/drive/MyDrive/logs")
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True
    print(f"✅ Google Drive подключён. Логи: {LOG_DIR}")
except Exception as e:
    # Если Drive недоступен — пишем локально в /content
    LOG_DIR = Path("/content/logs")
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = False
    print(f"⚠ Drive недоступен ({e}), логи в {LOG_DIR}")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID: {RUN_ID}")

Mounted at /content/drive
✅ Google Drive подключён. Логи: /content/drive/MyDrive/logs
RUN_ID: 20260603_051230


# НАСТРОЙКИ

In [3]:
load_dotenv(".env")

BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL_NAME = "YandexGPT-5-Lite-8B-instruct"
MAX_CONCURRENCY = 256
TEMPERATURE = 0
MAX_TEXT_LEN    = 1500

# ЗАГРУЗКА СЛОВАРЯ

In [4]:
def load_drug_terms(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[tuple[str, str | None]] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            term = (row.get("normalized_term") or "").strip()
            cat_raw = (row.get("category") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            category: str | None = cat_raw if cat_raw else None
            key = (term, category)
            if key in seen:
                continue
            seen.add(key)
            items.append({"term": term, "category": category})

    items.sort(key=lambda x: (x["term"], x["category"] or ""))
    return json.dumps(items, ensure_ascii=False)


def load_drug_terms_short(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[str] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cat_raw = (row.get("category") or "").strip()
            if cat_raw != "drugs":
                continue

            term = (row.get("normalized_term") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            if term in seen:
                continue
            seen.add(term)
            items.append({"term": term, "category": "drugs"})

    items.sort(key=lambda x: x["term"])
    return json.dumps(items, ensure_ascii=False)


DRUG_TERMS = load_drug_terms("illegal_terms_dictionary_edit.csv")
DRUG_TERMS_SHORT = load_drug_terms_short("illegal_terms_dictionary_edit.csv")

print(f"Загружено терминов в DRUG_TERMS: {len(json.loads(DRUG_TERMS))}")
print(f"Терминов в DRUG_TERMS_SHORT: {len(json.loads(DRUG_TERMS_SHORT))}")

Загружено терминов в DRUG_TERMS: 800
Терминов в DRUG_TERMS_SHORT: 104




# ЗАГРУЗКА ТЕСТОВОГО ДАТАСЕТА

In [5]:

df_test = pd.read_parquet("test.parquet").reset_index(drop=True)

def row_to_input_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":   f"{row['session_id']}___{idx}",  # составной ключ
        "text": text[:MAX_TEXT_LEN],
    }

input_jsons = [row_to_input_json(row, idx) for idx, row in df_test.iterrows()]

print(f"\nВсего записей в test: {len(input_jsons)}")
print(f"Уникальных session_id: {len({i['id'] for i in input_jsons})}")
print("\nПример:")
print(json.dumps(input_jsons[0], ensure_ascii=False, indent=2))

# Проверка уникальности ID — если сломается, значит session_id не уникален
assert len({i["id"] for i in input_jsons}) == len(input_jsons), (
    "session_id не уникален в test.parquet! "
    "Нужно использовать составной ключ (session_id + порядковый номер)."
)


Всего записей в test: 305
Уникальных session_id: 305

Пример:
{
  "id": "telegram-8412110593-adyoika-8412110593-NOn-52042546487-0880052307.91918c58-bfb1-1a6f-91d5-930278a7f694___0",
  "text": "Вопрос: /newNode_2;Тбилиси თბილისი\nОтвет: Выберите район \n\nაირჩიეთ რაიონი."
}


# Загрузка тренировочного датасета и выбор примеров для промпта

3 способа выбора примеров

In [6]:
df_train = pd.read_parquet("train.parquet").reset_index(drop=True)

# Задаём фиксированный seed для воспроизводимости
SEED_VALUE = 42
random.seed(SEED_VALUE)

# t — количество случайных примеров, которые нужно выбрать
t = 10


# ─────────────────────────────────────────────
# Подготовка данных
# ─────────────────────────────────────────────
def row_to_input_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":    f"{row['session_id']}___{idx}",
        "text":  text[:MAX_TEXT_LEN],
        "label": int(row["from_illegal_account"]),  # 0 = legal, 1 = illegal
    }

examples_jsons = [row_to_input_json(row, idx) for idx, row in df_train.iterrows()]

examples_id    = [item["id"]    for item in examples_jsons]
examples_text  = [item["text"]  for item in examples_jsons]
examples_label = [item["label"] for item in examples_jsons]

print(f"\nВсего записей в train : {len(examples_id)}")
print(f"Уникальных session_id : {len(set(examples_id))}")
print(f"Legal   (0)           : {examples_label.count(0)}")
print(f"Illegal (1)           : {examples_label.count(1)}")

# Проверка уникальности ID
assert len(set(examples_id)) == len(examples_id), (
    "session_id не уникален в train.parquet! "
    "Нужно использовать составной ключ (session_id + порядковый номер)."
)

# Индексы по классам
legal_indices   = [i for i, lbl in enumerate(examples_label) if lbl == 0]
illegal_indices = [i for i, lbl in enumerate(examples_label) if lbl == 1]

# ─────────────────────────────────────────────
# Вспомогательная функция вывода
# ─────────────────────────────────────────────
def print_samples(indices: list, title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for idx in sorted(indices):
        label_str = "illegal" if examples_label[idx] == 1 else "legal"
        print(f"\n[{label_str}]")
        #ID: {examples_id[idx]}")
        print(examples_text[idx])
        print("-" * 40)
# ─────────────────────────────────────────────
# Вариант 1 — t рандомных примеров
# ─────────────────────────────────────────────
def sample_random(t: int) -> list:
    return random.sample(range(len(examples_id)), t)

# ─────────────────────────────────────────────
# Вариант 2 — 1:1 (legal : illegal)
# n_illegal = t // 2  (округление вниз)
# n_legal   = t // 2  (округление вниз)
# итого примеров = n_legal + n_illegal (может быть < t, если t нечётное)
# ─────────────────────────────────────────────
def sample_balanced(t: int) -> list:
    n_illegal = t // 2
    n_legal   = t // 2
    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)
    return sampled_legal + sampled_illegal

# ─────────────────────────────────────────────
# Вариант 3 — 2:1 (legal : illegal)
# n_illegal = t // 3          (округление вниз)
# n_legal   = n_illegal * 2
# итого примеров = n_illegal * 3 (может быть < t, если t не кратно 3)
# ─────────────────────────────────────────────
def sample_2to1(t: int) -> list:
    n_illegal = t // 3
    n_legal   = n_illegal * 2
    actual_t  = n_legal + n_illegal   # фактическое число примеров
    #print(f"[Вариант 3] t={t} → n_legal={n_legal}, n_illegal={n_illegal}, итого={actual_t}")
    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)
    return sampled_legal + sampled_illegal



Всего записей в train : 748
Уникальных session_id : 748
Legal   (0)           : 182
Illegal (1)           : 566


# ПРОМПТЫ

In [7]:
SYSTEM_PROMPT = "Ты - помощник по классификации текста для задачи модерации на предмет упоминания наркотиков."
prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Прямое упоминание сущностей, связанных с наркотиками и наркоторговлей:
   - обменник, клад, кладмен, фасовка, закладка и т.д.

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова, ососбенно в названиях каналов и ботов через @:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - добавление лишних символов: DeaIler
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Подозрительные аббревиатуры и одиночные буквы латиницей в качестве вопроса:
   - Bbgg, Sh, I, CV GK j

6. Упоминания криптокошельков и криптовалют

7. Фразы с двойным дном и иносказания:
   -  Главное не забывать: счастье — это когда ты нашёл, а тебя нет!

8. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

9. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ПРИМЕРЫ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ:
{FEW_SHOT_EXAMPLES}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

In [8]:
# Запустите один раз после определения prompt_d
KNOWN_PLACEHOLDERS = {"FEW_SHOT_EXAMPLES", "DRUG_TERMS_SHORT", "INPUT_JSON"}

def escape_unknown_placeholders(text: str, known: set) -> str:
    """Экранирует {VAR} если VAR не в known."""
    def replacer(m):
        key = m.group(1)
        return f"{{{{{key}}}}}" if key not in known else m.group(0)
    return re.sub(r'\{(\w+)\}', replacer, text)

prompt_d = escape_unknown_placeholders(prompt_d, KNOWN_PLACEHOLDERS)

# Проверка
remaining = re.findall(r'\{(\w+)\}', prompt_d)
print("Оставшиеся плейсхолдеры:", remaining)
# Должно быть: ['FEW_SHOT_EXAMPLES', 'DRUG_TERMS_SHORT', 'INPUT_JSON']


Оставшиеся плейсхолдеры: ['DRUG_TERMS', 'FEW_SHOT_EXAMPLES', 'INPUT_JSON']


In [9]:
# ─────────────────────────────────────────────
# Построитель few-shot блока из списка индексов
# ─────────────────────────────────────────────
def build_few_shot_block(indices: list) -> str:
    """
    Формирует строку с примерами для вставки в промпт.
    Метка берётся из реального examples_label[idx].
    """
    blocks = []
    for idx in sorted(indices):
        label = bool(examples_label[idx] == 1)  # ← реальная метка
        label_str = "ILLEGAL" if label else "LEGAL"

        example_input = json.dumps(
            {"text": examples_text[idx]},
            ensure_ascii=False,
            indent=2,
        )
        example_output = json.dumps(
            {"has_drug_mention": label},          # ← без id, реальная метка
            ensure_ascii=False,
            indent=2,
        )
        blocks.append(
            f"[{label_str}]\n"
            f"ПРИМЕР ВХОДА:\n{example_input}\n\n"
            f"ПРИМЕР ВЫХОДА:\n{example_output}"
        )
    return "\n\n" + ("\n\n" + "─" * 40 + "\n\n").join(blocks) + "\n"


# ─────────────────────────────────────────────
# Построитель сообщений
# sample_fn — функция выборки, вызывается заново для каждого item
# ─────────────────────────────────────────────
def build_messages_d(item, sample_fn=None, use_dict: bool = True):
    """
    Для каждого item динамически вызывает sample_fn(t),
    чтобы пул few-shot примеров менялся при каждом вызове.
    """
    if sample_fn is not None:
        indices = sample_fn(t)                   # ← новая выборка на каждый item
        few_shot_block = build_few_shot_block(indices)
    else:
        few_shot_block = "(примеры не используются)"

    user_content = prompt_d.format(
        FEW_SHOT_EXAMPLES=few_shot_block,
        DRUG_TERMS_SHORT=DRUG_TERMS_SHORT if use_dict else "(словарь не используется)",
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


# ─────────────────────────────────────────────
# Варианты промптов — 6 штук
# sample_fn вызывается внутри lambda для каждого item заново
# ─────────────────────────────────────────────
PROMPT_VARIANTS = {
    "prompt_d_random_few_shot_wo_dict": lambda x: build_messages_d(
        x, sample_fn=sample_random, use_dict=False
    ),
    "prompt_d_balanced_few_shot_wo_dict": lambda x: build_messages_d(
        x, sample_fn=sample_balanced, use_dict=False
    ),
    "prompt_d_2to1_few_shot_wo_dict": lambda x: build_messages_d(
        x, sample_fn=sample_2to1, use_dict=False
    ),
    "prompt_d_random_few_shot_with_dict": lambda x: build_messages_d(
        x, sample_fn=sample_random, use_dict=True
    ),
    "prompt_d_balanced_few_shot_with_dict": lambda x: build_messages_d(
        x, sample_fn=sample_balanced, use_dict=True
    ),
    "prompt_d_2to1_few_shot_with_dict": lambda x: build_messages_d(
        x, sample_fn=sample_2to1, use_dict=True
    ),
}

# АСИНХРОННЫЕ ЗАПРОСЫ

In [10]:
async def send_one_request(client, model_name, messages):
    start = time.time()
    response = await client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=2048,
        temperature=0.0,
        seed=42,
    )
    end = time.time()
    return {
        "response":          response.choices[0].message.content,
        "time":              end - start,
        "prompt_tokens":     response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }


async def process_with_semaphore(client, model_name, messages):
    async with semaphore:
        return await send_one_request(client, model_name, messages)


def parse_response(raw: str, item_id: str):
    """Парсим JSON из ответа модели. Возвращает None если не удалось."""
    try:
        # Иногда модель оборачивает JSON в ```json ... ```
        cleaned = re.sub(r"```(?:json)?|```", "", raw).strip()
        data = json.loads(cleaned)
        return {
            "id":              str(data.get("id", item_id)),
            "has_drug_mention": bool(data.get("has_drug_mention", False)),
        }
    except Exception:
        print(f"  ⚠️  Не удалось распарсить ответ для id={item_id}: {raw[:100]}")
        return None


# ОЦЕНКА МЕТРИК

In [11]:
def evaluate_results(results_file: str, df_truth: pd.DataFrame, label: str):
    if not Path(results_file).exists():
        print(f"[{label}] файл {results_file} не найден, пропускаю")
        return None

    with open(results_file) as f:
        preds = json.load(f)

    df_pred = pd.DataFrame(preds)
    if df_pred.empty:
        print(f"[{label}] файл пустой, пропускаю")
        return None

    df_pred = df_pred.drop_duplicates(subset="id", keep="last")
    df_pred["id"] = df_pred["id"].astype(str)

    df_truth_local = df_truth.copy()
    df_truth_local["composite_id"] = (
            df_truth_local["session_id"].astype(str) + "___" +
            df_truth_local.index.astype(str)
    )

    # Диагностика выравнивания
    expected_ids = set(df_truth_local["session_id"])
    actual_ids   = set(df_pred["id"])
    missing = expected_ids - actual_ids
    extra   = actual_ids   - expected_ids
    if missing or extra:
        print(f"[{label}] ВНИМАНИЕ: пропущено id из truth: {len(missing)}, лишних id в pred: {len(extra)}")
        if extra:
            print(f"  пример лишних id: {list(extra)[:3]}")
        if missing:
            print(f"  пример пропущенных id: {list(missing)[:3]}")

    df_merged = df_truth_local.merge(
        df_pred, left_on="composite_id", right_on="id", how="inner"
    )

    if len(df_merged) == 0:
        print(f"[{label}] нет совпадений по session_id, пропускаю.")
        return None

    if len(df_merged) != len(df_truth_local):
        print(f"[{label}] предупреждение: смержилось {len(df_merged)} из {len(df_truth_local)} строк")

    y_true = (df_merged["message_label"] == "illegal").astype(int)
    y_pred = df_merged["has_drug_mention"].astype(int)

    metrics = {
        "version":   label,
        "n":         len(df_merged),
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

    print(f"\n=== Промпт {label} (n={metrics['n']}) ===")
    print(f"  accuracy : {metrics['accuracy']:.4f}")
    print(f"  precision: {metrics['precision']:.4f}")
    print(f"  recall   : {metrics['recall']:.4f}")
    print(f"  f1       : {metrics['f1']:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"\n  Матрица ошибок [строки=truth (legal, illegal), столбцы=pred]:")
    print(pd.DataFrame(
        cm,
        index=["truth_legal", "truth_illegal"],
        columns=["pred_legal", "pred_illegal"]
    ))

    print(f"\n  classification_report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["legal", "illegal"],
        zero_division=0
    ))

    return metrics



In [12]:
# ─────────────────────────────────────────────
# Настройка логирования
# ─────────────────────────────────────────────

class ColaFormatter(logging.Formatter):
    """Форматтер с цветами для вывода в ячейку Colab."""
    COLORS = {
        logging.DEBUG:    "\033[37m",    # белый
        logging.INFO:     "\033[36m",    # голубой
        logging.WARNING:  "\033[33m",    # жёлтый
        logging.ERROR:    "\033[31m",    # красный
        logging.CRITICAL: "\033[35m",    # фиолетовый
    }
    RESET = "\033[0m"

    def format(self, record):
        color = self.COLORS.get(record.levelno, self.RESET)
        record.levelname = f"{color}{record.levelname:<8}{self.RESET}"
        return super().format(record)


def setup_logger(
    name: str,
    log_file: str,
    console_level: int = logging.INFO,
    file_level: int    = logging.DEBUG,
) -> logging.Logger:
    """
    Создаёт логгер с тремя хэндлерами:
    - файл (все уровни, без цветов)
    - консоль Colab (INFO+, с цветами)
    - StringIO буфер (все уровни, для доступа из кода)
    """
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    # Очищаем хэндлеры при повторном вызове (важно для Colab)
    if logger.handlers:
        logger.handlers.clear()

    # Форматтеры
    file_fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_fmt = ColaFormatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    # 1. Файловый хэндлер
    file_path = LOG_DIR / log_file
    fh = logging.FileHandler(file_path, encoding="utf-8", mode="a")
    fh.setLevel(file_level)
    fh.setFormatter(file_fmt)
    logger.addHandler(fh)

    # 2. Консольный хэндлер (stdout для Colab)
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(console_level)
    ch.setFormatter(console_fmt)
    logger.addHandler(ch)

    # 3. StringIO буфер (для анализа логов в коде)
    buffer = StringIO()
    bh = logging.StreamHandler(buffer)
    bh.setLevel(logging.DEBUG)
    bh.setFormatter(file_fmt)
    logger.addHandler(bh)

    # Сохраняем буфер как атрибут логгера для доступа извне
    logger.buffer = buffer

    return logger


# Создаём логгеры
logger       = setup_logger("main",  f"run_{RUN_ID}.log",         console_level=logging.INFO)
api_logger   = setup_logger("api",   f"api_{RUN_ID}.log",         console_level=logging.WARNING)
parse_logger = setup_logger("parse", f"parse_errors_{RUN_ID}.log",console_level=logging.WARNING)

logger.info(f"Логирование настроено | RUN_ID={RUN_ID}")
logger.info(f"Логи сохраняются в: {LOG_DIR}")
logger.info(f"Google Drive: {'подключён' if USE_DRIVE else 'не используется'}")


05:12:50 | INFO     | main | Логирование настроено | RUN_ID=20260603_051230


INFO    :main:Логирование настроено | RUN_ID=20260603_051230


05:12:50 | INFO     | main | Логи сохраняются в: /content/drive/MyDrive/logs


INFO    :main:Логи сохраняются в: /content/drive/MyDrive/logs


05:12:50 | INFO     | main | Google Drive: подключён


INFO    :main:Google Drive: подключён


In [13]:
# ─────────────────────────────────────────────
# Утилиты для работы с логами
# ─────────────────────────────────────────────

def show_logs(logger_name: str = "main", tail: int = 50):
    """Выводит последние N строк из буфера логгера."""
    log = logging.getLogger(logger_name)
    if not hasattr(log, "buffer"):
        print("Буфер не найден")
        return
    lines = log.buffer.getvalue().splitlines()
    print(f"\n=== Последние {tail} строк лога [{logger_name}] ===")
    for line in lines[-tail:]:
        print(line)


def show_log_files():
    """Показывает все файлы логов и их размер."""
    print(f"\n=== Файлы логов в {LOG_DIR} ===")
    files = sorted(LOG_DIR.glob("*.log"))
    if not files:
        print("  (пусто)")
        return
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<45} {size_kb:>8.1f} KB")


def download_logs():
    """Скачивает все файлы логов в браузер (только Colab)."""
    try:
        from google.colab import files
        log_files = sorted(LOG_DIR.glob("*.log"))
        if not log_files:
            print("Нет файлов для скачивания")
            return
        for f in log_files:
            print(f"Скачиваем: {f.name}")
            files.download(str(f))
    except ImportError:
        print("Функция доступна только в Google Colab")


def clear_log_buffer(logger_name: str = "main"):
    """Очищает StringIO буфер логгера."""
    log = logging.getLogger(logger_name)
    if hasattr(log, "buffer"):
        log.buffer.truncate(0)
        log.buffer.seek(0)
        print(f"Буфер [{logger_name}] очищен")


# ГЛАВНАЯ ФУНКЦИЯ

In [14]:
# ─────────────────────────────────────────────
# Асинхронные функции с логированием
# ─────────────────────────────────────────────

async def send_one_request(client, model_name, messages, item_id: str = "unknown"):
    api_logger.debug(
        f"[{item_id}] → Запрос | "
        f"prompt_len={sum(len(m['content']) for m in messages)}"
    )
    start = time.time()
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=messages,
            max_tokens=2048,
            temperature=0.0,
            seed=42,
        )
        elapsed = time.time() - start
        content = response.choices[0].message.content

        api_logger.debug(
            f"[{item_id}] ← Ответ | "
            f"time={elapsed:.2f}s | "
            f"prompt_tokens={response.usage.prompt_tokens} | "
            f"completion_tokens={response.usage.completion_tokens} | "
            f"preview={content[:60].replace(chr(10),' ')!r}"
        )
        return {
            "response":          content,
            "time":              elapsed,
            "prompt_tokens":     response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        }

    except RateLimitError as e:
        elapsed = time.time() - start
        api_logger.warning(f"[{item_id}] RateLimitError | time={elapsed:.2f}s | {e}")
        raise

    except APIConnectionError as e:
        elapsed = time.time() - start
        api_logger.error(f"[{item_id}] APIConnectionError | time={elapsed:.2f}s | {e}")
        raise

    except APIStatusError as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] APIStatusError | "
            f"time={elapsed:.2f}s | "
            f"status={e.status_code} | "
            f"body={str(e.body)[:200]}"
        )
        raise

    except Exception as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] UnexpectedError | "
            f"time={elapsed:.2f}s | "
            f"{type(e).__name__}: {e}"
        )
        raise


async def process_with_semaphore(client, model_name, messages, item_id: str = "unknown"):
    async with semaphore:
        return await send_one_request(client, model_name, messages, item_id=item_id)


def parse_response(raw: str, item_id: str):
    try:
        cleaned = re.sub(r"```(?:json)?|```", "", raw).strip()
        data    = json.loads(cleaned)
        result  = {
            "id":               str(data.get("id", item_id)),
            "has_drug_mention": bool(data.get("has_drug_mention", False)),
        }
        api_logger.debug(
            f"[{item_id}] Парсинг OK | "
            f"has_drug_mention={result['has_drug_mention']}"
        )
        return result

    except json.JSONDecodeError as e:
        parse_logger.error(
            f"[{item_id}] JSONDecodeError: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None

    except Exception as e:
        parse_logger.error(
            f"[{item_id}] ParseError: {type(e).__name__}: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None


In [15]:
async def run_variant(client, variant_name, items):
    """Классифицирует все items одним промптом, сохраняет результаты в JSON."""
    print(f"\n{'='*50}")
    print(f"Запускаем вариант: {variant_name}")
    print(f"Записей: {len(items)}")

    builder = PROMPT_VARIANTS[variant_name]
    tasks = [
        asyncio.create_task(
            process_with_semaphore(client, MODEL_NAME, builder(item))
        )
        for item in items
    ]

    start = time.time()
    raw_results = await tqdm.gather(*tasks, desc=variant_name)
    end = time.time()

    # Статистика времени
    valid = [r for r in raw_results if not isinstance(r, Exception)]
    print(f"Всего времени: {end - start:.2f} сек")
    print(f"Среднее время на запрос: {sum(x['time'] for x in valid) / len(valid):.2f} сек")
    print(f"Средний prompt_tokens: {sum(x['prompt_tokens'] for x in valid) / len(valid):.0f}")
    print(f"Средний completion_tokens: {sum(x['completion_tokens'] for x in valid) / len(valid):.0f}")

    # Парсим ответы
    parsed = []
    for item, raw in zip(items, raw_results):
        if isinstance(raw, Exception):
            print(f"  ⚠️  Ошибка запроса для id={item['id']}: {raw}")
            continue
        result = parse_response(raw["response"], item["id"])
        if result:
            parsed.append(result)

    # Сохраняем результаты
    output_file = f"results_{variant_name}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed, f, ensure_ascii=False, indent=2)
    print(f"Сохранено {len(parsed)} результатов → {output_file}")

    return parsed


async def main():
    global semaphore
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    client = AsyncOpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
    )
    print(f"Модель: {MODEL_NAME}")
    print(f"Записей для классификации: {len(input_jsons)}")

    # Шаг 1 — классификация всеми вариантами промптов
    for variant_name in PROMPT_VARIANTS:
        await run_variant(client, variant_name, input_jsons)

    # Шаг 2 — считаем метрики
    print(f"\n{'='*50}")
    print("ИТОГОВЫЕ МЕТРИКИ")
    print(f"{'='*50}")

    all_metrics = []
    for variant_name in PROMPT_VARIANTS:
        m = evaluate_results(
            f"results_{variant_name}.json",
            df_test,
            variant_name
        )
        if m is not None:
            all_metrics.append(m)

    if all_metrics:
        print("\n=== Сводная таблица ===")
        print(pd.DataFrame(all_metrics).set_index("version").round(4))


if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())


Модель: YandexGPT-5-Lite-8B-instruct
Записей для классификации: 305

Запускаем вариант: prompt_d_random_few_shot_wo_dict
Записей: 305


prompt_d_random_few_shot_wo_dict:   0%|          | 0/305 [00:00<?, ?it/s]DEBUG:api:[unknown] → Запрос | prompt_len=6477
DEBUG:api:[unknown] → Запрос | prompt_len=6824
DEBUG:api:[unknown] → Запрос | prompt_len=6987
DEBUG:api:[unknown] → Запрос | prompt_len=7044
DEBUG:api:[unknown] → Запрос | prompt_len=6447
DEBUG:api:[unknown] → Запрос | prompt_len=7122
DEBUG:api:[unknown] → Запрос | prompt_len=6773
DEBUG:api:[unknown] → Запрос | prompt_len=6721
DEBUG:api:[unknown] → Запрос | prompt_len=7370
DEBUG:api:[unknown] → Запрос | prompt_len=6448
DEBUG:api:[unknown] → Запрос | prompt_len=6093
DEBUG:api:[unknown] → Запрос | prompt_len=6850
DEBUG:api:[unknown] → Запрос | prompt_len=7543
DEBUG:api:[unknown] → Запрос | prompt_len=7234
DEBUG:api:[unknown] → Запрос | prompt_len=6731
DEBUG:api:[unknown] → Запрос | prompt_len=7146
DEBUG:api:[unknown] → Запрос | prompt_len=6401
DEBUG:api:[unknown] → Запрос | prompt_len=6294
DEBUG:api:[unknown] → Запрос | prompt_len=6827
DEBUG:api:[unknown] → Запрос | pro

Всего времени: 31.67 сек
Среднее время на запрос: 17.26 сек
Средний prompt_tokens: 2336
Средний completion_tokens: 16


DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-6505442291.b556d0a7-7b3c-d68d-4cde-91d121a348f3___250] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-6505442291.b556d0a7-7b3c-d68d-4cde-91d121a348f3___251] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-183263525-massatela1_startapy_-183263525-gul-14644976955-2827676841.2efee0d3-d2d8-ec94-1411-a580109e6ccf___252] Парсинг OK | has_drug_mention=False
DEBUG:api:[incoming_webim2-591417391-gentle_alya-591417391-AqL-43674851389-2281190.8d56c49b-25cf-9177-87d8-771b77247746___253] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-8412110593-adyoika-8412110593-NOn-52042546487-2188501406.e3d3bb43-0bfb-c7c3-4f31-793b79703d89___254] Парсинг OK | has_drug_mention=True
DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-6505442291.b556d0a7-7b3c-d68d-4cde-91d121a348f3___255] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-322189240-onl

Сохранено 305 результатов → results_prompt_d_random_few_shot_wo_dict.json

Запускаем вариант: prompt_d_balanced_few_shot_wo_dict
Записей: 305


...

Сохранено 305 результатов → results_prompt_d_balanced_few_shot_with_dict.json

Запускаем вариант: prompt_d_2to1_few_shot_with_dict
Записей: 305


prompt_d_2to1_few_shot_with_dict:   0%|          | 0/305 [00:00<?, ?it/s]DEBUG:api:[unknown] → Запрос | prompt_len=6359
DEBUG:api:[unknown] → Запрос | prompt_len=6607
DEBUG:api:[unknown] → Запрос | prompt_len=6410
DEBUG:api:[unknown] → Запрос | prompt_len=7062
DEBUG:api:[unknown] → Запрос | prompt_len=6191
DEBUG:api:[unknown] → Запрос | prompt_len=6486
DEBUG:api:[unknown] → Запрос | prompt_len=7049
DEBUG:api:[unknown] → Запрос | prompt_len=6790
DEBUG:api:[unknown] → Запрос | prompt_len=6822
DEBUG:api:[unknown] → Запрос | prompt_len=6331
DEBUG:api:[unknown] → Запрос | prompt_len=6036
DEBUG:api:[unknown] → Запрос | prompt_len=7019
DEBUG:api:[unknown] → Запрос | prompt_len=6043
DEBUG:api:[unknown] → Запрос | prompt_len=6614
DEBUG:api:[unknown] → Запрос | prompt_len=6292
DEBUG:api:[unknown] → Запрос | prompt_len=7030
DEBUG:api:[unknown] → Запрос | prompt_len=6485
DEBUG:api:[unknown] → Запрос | prompt_len=7216
DEBUG:api:[unknown] → Запрос | prompt_len=6150
DEBUG:api:[unknown] → Запрос | pro

Всего времени: 28.76 сек
Среднее время на запрос: 15.87 сек
Средний prompt_tokens: 2221
Средний completion_tokens: 16


DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-6505442291.b556d0a7-7b3c-d68d-4cde-91d121a348f3___196] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-8412110593-adyoika-8412110593-NOn-52042546487-914369141.371d3f9c-5504-46b3-3666-5f84387e83b3___197] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-8412110593-adyoika-8412110593-NOn-52042546487-914369141.371d3f9c-5504-46b3-3666-5f84387e83b3___198] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-2105914475.c74b6d55-ed60-69e4-71ce-adf67ce9e8fc___199] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-8412110593-adyoika-8412110593-NOn-52042546487-2188501406.e3d3bb43-0bfb-c7c3-4f31-793b79703d89___200] Парсинг OK | has_drug_mention=False
DEBUG:api:[telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-9852144909.61138f1d-865b-241b-63a2-59e10158c078___201] Парсинг OK | has_drug_mention=True
DEBUG:api:[telegram-8412110593-adyoika-8412110593-NO

Сохранено 305 результатов → results_prompt_d_2to1_few_shot_with_dict.json

ИТОГОВЫЕ МЕТРИКИ
[prompt_d_random_few_shot_wo_dict] ВНИМАНИЕ: пропущено id из truth: 39, лишних id в pred: 305
  пример лишних id: ['telegram-8412110593-adyoika-8412110593-NOn-52042546487-914369141.371d3f9c-5504-46b3-3666-5f84387e83b3___146', 'incoming_webim2-591417391-gentle_alya-591417391-AqL-43674851389-0734620.5ac8c6be-1330-724d-47da-e64a573b391c___30', 'telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-8352479131.0498ccba-1a56-1773-6ca7-fd3df157c875___166']
  пример пропущенных id: ['telegram-8412110593-adyoika-8412110593-NOn-52042546487-9457988541.16f943a7-4421-564f-64c2-7b34cae3efb7', 'incoming_webim2-591417391-gentle_alya-591417391-AqL-43674851389-0734620.5ac8c6be-1330-724d-47da-e64a573b391c', 'telegram-183263525-cargo-183263525-gbb-81963901972-2741213963.e205119d-7628-30b1-ccd2-ad6657057dd6']

=== Промпт prompt_d_random_few_shot_wo_dict (n=305) ===
  accuracy : 0.6689
  precision: 0.5057
  recal

In [16]:
download_logs()

Скачиваем: api_20260601_192202.log


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачиваем: api_20260602_015456.log


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачиваем: run_20260603_051230.log


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''
# Последние 30 строк основного лога
show_logs("main", tail=30)

# Все ошибки парсинга
show_logs("parse", tail=100)

# Список файлов логов
show_log_files()

# Скачать логи в браузер
download_logs()
'''